## **The Correlation pitfalls, Illustrated with  SHAP**


**Toy structural model:** $X_1$ drives $X_2$ and $X_3$, which both feed $Y$ directly.
$X_1$ itself has **no direct edge into $Y$**.

```
X1 ---> X2 ---> Y
 \-----> X3 ---/
```

$$
X_1 \sim \mathcal{N}(0,1), \qquad X_2 = a X_1 + \varepsilon_2, \qquad X_3 = b X_1 +
\varepsilon_3, \qquad Y = c X_2 + d X_3 + \varepsilon_Y
$$

Since $Y$'s only parents are $X_2$ and $X_3$: **once $X_2$ and $X_3$ are known exactly,
$X_1$ carries no further information about $Y$** ($X_1 \perp Y \mid X_2, X_3$). This
notebook shows, with the real `shap` library, that this makes classical (marginal) Shapley
values assign $X_1$ essentially **zero** credit — even though intervening on $X_1$ changes
$Y$ substantially, through $X_2$ and $X_3$.


In [ ]:
!pip install -q shap


In [ ]:
import numpy as np
import math
from itertools import combinations
import shap
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(0)
n = 5000
a, b, c, d = 1.5, -1.0, 2.0, 1.0   # structural coefficients
sigma = 0.5                        # noise std

X1 = rng.normal(0, 1, n)
X2 = a*X1 + rng.normal(0, sigma, n)
X3 = b*X1 + rng.normal(0, sigma, n)
Y  = c*X2 + d*X3 + rng.normal(0, sigma, n)
X = np.column_stack([X1, X2, X3])

print(f"Structural coefficients: a={a}, b={b}, c={c}, d={d}")


Structural coefficients: a=1.5, b=-1.0, c=2.0, d=1.0


## **Step 1 : Fit the predictive model**

We fit a linear regression $f(x_1,x_2,x_3)\to y$ on the observed data. Because $X_1 \perp
Y \mid (X_2,X_3)$ by construction, the population-optimal coefficient on $X_1$ is exactly
zero -- the fitted model should recover this.


In [ ]:
model = LinearRegression().fit(X, Y)
print("Fitted coefficients [X1, X2, X3]:", np.round(model.coef_, 4))
print("(X1's coefficient is ~0: Y doesn't need X1 once X2, X3 are known)")


Fitted coefficients [X1, X2, X3]: [-0.02    1.9962  0.9886]
(X1's coefficient is ~0: Y doesn't need X1 once X2, X3 are known)


## **Step 2 : Classical (marginal) SHAP, using the `shap` library**

We compute genuine SHAP values (not a hand-rolled approximation) for one specific instance,
using `shap.LinearExplainer` with a background dataset for marginalization.


In [ ]:
background = shap.sample(X, 200, random_state=0)
explainer = shap.LinearExplainer(model, background)

x_instance = X[0:1]
shap_values = explainer(x_instance)

print("Instance: X1=%.4f, X2=%.4f, X3=%.4f" % tuple(x_instance[0]))
print("\nClassical (marginal) SHAP values:")
print("phi_1 (X1):", round(shap_values.values[0,0], 4))
print("phi_2 (X2):", round(shap_values.values[0,1], 4))
print("phi_3 (X3):", round(shap_values.values[0,2], 4))
print("\nCheck -- base value + sum(phi) should equal the model's prediction:")
print("base + sum(phi) =", round(shap_values.base_values[0] + shap_values.values[0].sum(), 4),
      " | model prediction =", round(model.predict(x_instance)[0], 4))


Instance: X1=0.1257, X2=0.0986, X3=0.1190

Classical (marginal) SHAP values:
phi_1 (X1): -0.0037
phi_2 (X2): 0.3693
phi_3 (X3): 0.083

Check -- base value + sum(phi) should equal the model's prediction:
base + sum(phi) = 0.3101  | model prediction = 0.3101


**What you should see:** $\phi_1 \approx 0$ (up to fitting/sampling noise), while $\phi_2$
and $\phi_3$ carry essentially all the credit. This is the *correct* answer to the question
classical SHAP actually asks ("how much does the *fitted model* $f$ rely on each input?") --
but it is easy to misread this as "$X_1$ doesn't matter," which is false about the
underlying causal system.
